In [5]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (path / "src" / "data_loader.py").is_file()
        and (path / "src" / "features.py").is_file()
        and (path / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root was not found. Run the notebook from the project "
        "directory or one of its subdirectories."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {PROJECT_ROOT / 'data' / 'raw'}")

Project root: C:\Users\Nastya\Documents\retail-sales-forecasting
Raw data directory: C:\Users\Nastya\Documents\retail-sales-forecasting\data\raw


In [6]:
from src.data_loader import load_data
from src import features as ft

importlib.reload(ft)

df = load_data()

print(f"Dataset shape: {df.shape}")
print(f"Number of stores: {df['Store'].nunique()}")
print(f"Date range: {df['Date'].min().date()} — {df['Date'].max().date()}")
print(f"Number of unique dates: {df['Date'].nunique()}")

display(df.head())
display(df.dtypes.rename("dtype").to_frame())

Dataset shape: (1017209, 20)
Number of stores: 1115
Date range: 2013-01-01 — 2015-07-31
Number of unique dates: 942


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistanceMissing,CompetitionInfoMissing
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,0,0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",0,0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",0,0
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN,0,0
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN,0,0


,dtype
Store,int64
DayOfWeek,int64
Date,datetime64[us]
Sales,int64
Customers,int64
Open,int64
Promo,int64
StateHoliday,string
SchoolHoliday,int64
StoreType,str


## Kontrola jakości danych przed tworzeniem cech
Sprawdzamy duplikaty, brakujące wartości oraz spójność klucza Store–Date. Open - Sales.
Braków w metadanych nie zastępujemy automatycznie bez rozpoznania ich znaczenia.

In [8]:
print(f"Full-row duplicates: {df.duplicated().sum()}")
print(f"Duplicate Store-Date rows: {df.duplicated(['Store', 'Date']).sum()}")

missing_values = (
    df.isna().sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)
display(missing_values.loc[missing_values["missing_count"] > 0])

assert df["Store"].notna().all(), "Missing Store IDs found."
assert df["Date"].notna().all(), "Missing dates found."
assert not df.duplicated(["Store", "Date"]).any(), "Duplicate Store-Date rows found."

Full-row duplicates: 0
Duplicate Store-Date rows: 0


,missing_count
PromoInterval,508031
Promo2SinceWeek,508031
Promo2SinceYear,508031
CompetitionOpenSinceMonth,323348
CompetitionOpenSinceYear,323348


In [12]:
print("Zero-sales observations:", int(df["Sales"].eq(0).sum()))
print(
    "Open-store observations with zero sales:",
    int((df["Open"].eq(1) & df["Sales"].eq(0)).sum()),
)

display(
    df.loc[
        df["Open"].eq(1) & df["Sales"].eq(0),
        ["Store", "Date", "Open", "Sales", "Customers", "Promo"],
    ].head(10)
)



Zero-sales observations: 172871
Open-store observations with zero sales: 54


,Store,Date,Open,Sales,Customers,Promo
86825,971,2015-05-15,1,0,0,0
142278,674,2015-03-26,1,0,0,0
196938,699,2015-02-05,1,0,0,1
322053,708,2014-10-01,1,0,0,1
330176,357,2014-09-22,1,0,0,0
340348,227,2014-09-11,1,0,0,0
340860,835,2014-09-11,1,0,0,0
341795,835,2014-09-10,1,0,0,0
346232,548,2014-09-05,1,0,0,1
346734,28,2014-09-04,1,0,0,1


## Cechy kalendarzowe
Cechy kalendarzowe są wyprowadzane z Date. 
Nie korzystają z przyszłych wartości Sales, dlatego mogą być znane z wyprzedzeniem na cały 30-dniowy horyzont.

In [13]:
features_df = ft.add_calendar_features(df)

print(f"Shape before calendar features: {df.shape}")
print(f"Shape after calendar features:  {features_df.shape}")

calendar_columns = [
    "Store", "Date", "Year", "Month", "Day", "Weekday",
    "WeekOfYear", "Quarter", "DayOfYear",
    "MonthSin", "MonthCos", "WeekdaySin", "WeekdayCos",
]
display(features_df[calendar_columns].head(10))

Shape before calendar features: (1017209, 20)
Shape after calendar features:  (1017209, 31)


,Store,Date,Year,Month,Day,Weekday,WeekOfYear,Quarter,DayOfYear,MonthSin,MonthCos,WeekdaySin,WeekdayCos
0,1,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
1,2,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
2,3,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
3,4,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
4,5,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
5,6,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
6,7,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
7,8,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
8,9,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521
9,10,2015-07-31,2015,7,31,5,31,3,212,-0.5,-0.866025,-0.974928,-0.222521


In [14]:
# Check weekday consistency, month range
weekday_check = (
    features_df["Date"].dt.dayofweek.add(1).eq(features_df["Weekday"]).all()
)
month_check = features_df["Month"].between(1, 12).all()
weekday_range_check = features_df["Weekday"].between(1, 7).all()

print("Weekday consistency:", weekday_check)
print("Month range:", month_check)
print("Weekday range:", weekday_range_check)

assert weekday_check
assert month_check
assert weekday_range_check

Weekday consistency: True
Month range: True
Weekday range: True


## Lagi sprzedaży i statystyki kroczące
Tworzymy lagi Sales dla 1, 7, 14 i 28 dni oraz średnią i odchylenie standardowe z poprzednich 7, 14 i 28 obserwacji.

Zasady:
- cechy są obliczane osobno dla każdego sklepu;
- dane są uporządkowane chronologicznie;
- rolling mean i rolling std nie obejmują bieżącego dnia — przed obliczeniem stosowany jest shift(1);
- braki dat nie są sztucznie uzupełniane;
- po przerwie w historii obliczenia zaczynają się od nowego ciągłego segmentu;
- zerowa sprzedaż pozostaje wartością obserwowaną i nie jest automatycznie zamieniana na brak danych.

In [24]:
features_df = ft.add_sales_lag_features(
    features_df,
    lags=(1, 7, 14, 28),
    windows=(7, 14, 28),
)

print(f"Shape after feature engineering: {features_df.shape}")

sales_feature_columns = [
    "Store", "Date", "Open", "Sales",
    "segment_id",
    "Sales_lag_1", "Sales_lag_7", "Sales_lag_14", "Sales_lag_28",
    "Sales_roll_mean_7", "Sales_roll_std_7",
    "Sales_roll_mean_14", "Sales_roll_std_14",
    "Sales_roll_mean_28", "Sales_roll_std_28",
]
display(features_df[sales_feature_columns].head(30))

Shape after feature engineering: (1017209, 42)


,Store,Date,Open,Sales,segment_id,Sales_lag_1,Sales_lag_7,Sales_lag_14,Sales_lag_28,Sales_roll_mean_7,Sales_roll_std_7,Sales_roll_mean_14,Sales_roll_std_14,Sales_roll_mean_28,Sales_roll_std_28
0,1,2015-07-31,1,5263,1,5020.0,3706.0,4852.0,4665.0,4140.714286,1965.230157,3816.357143,1763.345187,3757.928571,1676.185063
1,2,2015-07-31,1,6064,1,5567.0,3854.0,4518.0,4838.0,4376.142857,2420.865508,4002.857143,2041.576610,4110.035714,2030.159061
2,3,2015-07-31,1,8314,1,8977.0,5080.0,6679.0,8149.0,6073.714286,3301.712218,5426.214286,2786.854394,5695.000000,2832.551931
3,4,2015-07-31,1,13995,1,10387.0,8322.0,10514.0,11543.0,8661.714286,3968.863007,8292.642857,3689.440433,8621.464286,3832.296125
4,5,2015-07-31,1,4822,1,4943.0,3815.0,4355.0,4475.0,4261.285714,2504.298086,3662.857143,2089.427815,3824.285714,2115.879204
5,6,2015-07-31,1,5651,1,4790.0,3804.0,4422.0,4594.0,3903.285714,1963.117395,3545.142857,1717.926293,3708.821429,1787.325336
6,7,2015-07-31,1,15344,1,11560.0,9203.0,10339.0,11976.0,9517.428571,4944.184458,8636.714286,4263.950812,8592.678571,4101.358014
7,8,2015-07-31,1,8492,1,8420.0,4917.0,6728.0,6535.0,5587.142857,3116.251136,5134.285714,2660.159580,5333.500000,2757.209224
8,9,2015-07-31,1,8565,1,7539.0,5464.0,6845.0,8363.0,6501.285714,3407.860605,5963.285714,2906.540276,6040.178571,2859.131804
9,10,2015-07-31,1,7185,1,6186.0,6152.0,5356.0,5915.0,5132.428571,2386.085627,5114.000000,2244.761183,5046.964286,2208.395403


In [17]:
store_id = 100

store_check = (
    features_df.loc[
        features_df["Store"] == store_id,
        [
            "Store",
            "Date",
            "Sales",
            "Sales_lag_1",
            "Sales_lag_7",
            "Sales_roll_mean_7",
        ],
    ]
    .sort_values("Date")
)

display(store_check.head(20))

,Store,Date,Sales,Sales_lag_1,Sales_lag_7,Sales_roll_mean_7
1016194,100,2013-01-01,0,NaN,NaN,NaN
1015079,100,2013-01-02,6969,0.0,NaN,NaN
1013964,100,2013-01-03,7398,6969.0,NaN,NaN
1012849,100,2013-01-04,7129,7398.0,NaN,NaN
1011734,100,2013-01-05,7973,7129.0,NaN,NaN
1010619,100,2013-01-06,0,7973.0,NaN,NaN
1009504,100,2013-01-07,10609,0.0,NaN,NaN
1008389,100,2013-01-08,6816,10609.0,0.0,5725.428571
1007274,100,2013-01-09,7883,6816.0,6969.0,6699.142857
1006159,100,2013-01-10,7614,7883.0,7398.0,6829.714286


In [18]:
store_id = 100

store_check = (
    features_df.loc[
        (features_df["Store"] == store_id)
        & (features_df["Date"] >= "2014-06-25")
        & (features_df["Date"] <= "2015-02-10"),
        [
            "Store",
            "Date",
            "Sales",
            "Sales_lag_1",
            "Sales_lag_7",
            "Sales_roll_mean_7",
        ],
    ]
    .sort_values("Date")
)

display(store_check)

,Store,Date,Sales,Sales_lag_1,Sales_lag_7,Sales_roll_mean_7
414094,100,2014-06-25,6306,7200.0,8649.0,5878.000000
412979,100,2014-06-26,6129,6306.0,0.0,5543.285714
411864,100,2014-06-27,7450,6129.0,9953.0,6418.857143
410749,100,2014-06-28,7407,7450.0,8049.0,6061.285714
409634,100,2014-06-29,0,7407.0,0.0,5969.571429
408519,100,2014-06-30,12656,0.0,7295.0,5969.571429
235364,100,2015-01-01,0,NaN,NaN,NaN
234249,100,2015-01-02,8797,0.0,NaN,NaN
233134,100,2015-01-03,9166,8797.0,NaN,NaN
232019,100,2015-01-04,0,9166.0,NaN,NaN


## Kontrola poprawności lagów i rolling features

Weryfikujemy, czy Sales_lag_1 odpowiada poprzedniej obserwacji w obrębie sklepu i ciągłego segmentu oraz czy statystyki kroczące nie wykorzystują bieżącej wartości docelowej.

In [20]:
ordered = features_df.sort_values(["Store", "Date"]).copy()

expected_lag_1 = (
    ordered.groupby(["Store", "segment_id"], sort=False)["Sales"].shift(1)
)
lag_1_check = ordered["Sales_lag_1"].reset_index(drop=True).equals(
    expected_lag_1.reset_index(drop=True)
)

# Recalculate the 7-observation rolling mean from strictly prior sales.
expected_roll_mean_7 = (
    ordered.groupby(["Store", "segment_id"], sort=False)["Sales"]
    .transform(lambda s: s.shift(1).rolling(window=7, min_periods=7).mean())
)
rolling_mean_7_check = np.allclose(
    ordered["Sales_roll_mean_7"].to_numpy(dtype=float),
    expected_roll_mean_7.to_numpy(dtype=float),
    equal_nan=True,
)

print("Lag-1 check:", lag_1_check)
print("Rolling mean 7 check:", rolling_mean_7_check)

assert lag_1_check
assert rolling_mean_7_check

Lag-1 check: True
Rolling mean 7 check: True


## Braki w cechach historycznych
Braki na początku historii sklepu lub segmentu są oczekiwane: nie ma wtedy wystarczającej liczby wcześniejszych obserwacji do obliczenia laga albo statystyki kroczącej. Nie wypełniamy ich automatycznie zerami.

In [21]:
lag_columns = [
    col for col in features_df.columns
    if col.startswith("Sales_lag_") or col.startswith("Sales_roll_")
]

missing_feature_summary = pd.DataFrame({
    "missing_count": features_df[lag_columns].isna().sum(),
    "missing_pct": (features_df[lag_columns].isna().mean() * 100).round(2),
})
display(missing_feature_summary.sort_values("missing_count", ascending=False))

,missing_count,missing_pct
Sales_roll_std_28,36260,3.56
Sales_lag_28,36260,3.56
Sales_roll_mean_28,36260,3.56
Sales_roll_mean_14,18130,1.78
Sales_roll_std_14,18130,1.78
Sales_lag_14,18130,1.78
Sales_roll_std_7,9065,0.89
Sales_lag_7,9065,0.89
Sales_roll_mean_7,9065,0.89
Sales_lag_1,1295,0.13


## Kontrola ciągłości historii
Brakujące daty są sprawdzane osobno dla każdego sklepu. 
Funkcja tworząca cechy przypisuje nowy segment_id po przerwie w kolejnych dniach, aby lagi i rolling features nie przekraczały luk w historii.

In [32]:
date_gaps = (
    features_df.sort_values(["Store", "Date"])
    .assign(
        date_diff=lambda x: (
            x.groupby("Store", sort=False)["Date"].diff().dt.days
        )
    )
)

gap_summary = (
    date_gaps.loc[date_gaps["date_diff"].gt(1)]
    .groupby("Store")
    .agg(
        gap_count=("date_diff", "size"),
        max_gap_days=("date_diff", "max"),
    )
    .sort_values(["gap_count", "max_gap_days"], ascending=False)
)

print("Number of stores with gaps:", len(gap_summary))
display(gap_summary.head(10))
display(gap_summary.tail(10))

Number of stores with gaps: 180


,gap_count,max_gap_days
Store,,
13,1,185.0
20,1,185.0
22,1,185.0
32,1,185.0
36,1,185.0
41,1,185.0
46,1,185.0
51,1,185.0
52,1,185.0


,gap_count,max_gap_days
Store,,
1056,1,185.0
1065,1,185.0
1067,1,185.0
1080,1,185.0
1092,1,185.0
1094,1,185.0
1102,1,185.0
1104,1,185.0
1107,1,185.0


In [34]:
assert features_df[["Store", "Date"]].notna().all().all()
assert not features_df.duplicated(["Store", "Date"]).any()
assert features_df["Date"].dtype == "datetime64[ns]" or pd.api.types.is_datetime64_any_dtype(features_df["Date"])
assert features_df["Sales"].notna().all()
assert features_df["segment_id"].notna().all()

print("Final shape:", features_df.shape)
print("Stores:", features_df["Store"].nunique())
print("Date range:", features_df["Date"].min().date(), "—", features_df["Date"].max().date())
print("Unique Store-Date keys:", features_df[["Store", "Date"]].drop_duplicates().shape[0])
print("Target column: Sales")
print("Final chronological split: not performed in this notebook.")

Final shape: (1017209, 42)
Stores: 1115
Date range: 2013-01-01 — 2015-07-31
Unique Store-Date keys: 1017209
Target column: Sales
Final chronological split: not performed in this notebook.


## Wnioski z kontroli jakości i tworzenia cech

Po utworzeniu cech historycznych zbiór danych zawiera 1 017 209 obserwacji, 42 kolumny i 1 115 sklepów. Zakres dat obejmuje okres od 1 stycznia 2013 r. do 31 lipca 2015 r.

Kontrole potwierdziły brak duplikatów dla klucza `Store`–`Date`, brak brakujących wartości w kolumnach `Store`, `Date` i `Sales` oraz poprawne przypisanie identyfikatorów segmentów czasowych.

Braki w lagach i statystykach kroczących są związane z niewystarczającą liczbą wcześniejszych obserwacji. Ich udział wzrasta wraz z długością laga lub okna i wynosi maksymalnie 3,56% dla cech opartych na 28 wcześniejszych obserwacjach. Braki te nie powinny być automatycznie zastępowane zerami, ponieważ brak historii nie jest równoważny zerowej sprzedaży.

Wykryto również przerwy w historii 180 sklepów. Podział historii na ciągłe segmenty ogranicza ryzyko obliczania cech na podstawie obserwacji oddzielonych brakującymi datami. 


In [36]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "features.parquet"
features_df.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows: {len(features_df):,}")
print(f"Columns: {len(features_df.columns):,}")

Saved: C:\Users\Nastya\Documents\retail-sales-forecasting\data\processed\features.parquet
Rows: 1,017,209
Columns: 42
